# Эксперимент 18 v3: SSL Domain Adaptation (60 эпох, warm-start с v2)

## Что изменилось по сравнению с v2

В v2 SSL loss не достиг плато к эпохе 30 — всё ещё монотонно снижался (0.411 → 0.363).
Здесь мы:

1. **Загружаем backbone после v2** (`dinov2_ssl_exp18v2.pt`) как тёплый старт
2. **Обучаем 60 эпох** (вместо 30) с новым косинусным расписанием
3. **Полный checkpoint-resume** — при обрыве продолжаем с последней сохранённой эпохи

| Версия | batch | Негативов | Epochs | Projection head | Warm-start |
|---|---|---|---|---|---|
| v1 | 2 | 2 | 15 | ❌ | — |
| v2 | 4 | 6 | 30 | ✅ | DINOv2 pretrained |
| **v3** | **4** | **6** | **60** | **✅** | **v2 backbone (30 эп. SSL)** |

## 1. Импорты и конфигурация

In [ ]:
import os, random, json, pickle, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchmetrics import JaccardIndex
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

print(f'PyTorch: {torch.__version__}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Гб')

DATA_DIR      = 'D:/VKR/VKR/DATA_DIR/'
TRAIN_CSV     = os.path.join(DATA_DIR, 'train.csv')
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train_images/')
save_dir      = 'D:/VKR/VKR/dino_heads'
os.makedirs(save_dir, exist_ok=True)

# ── Архитектура ───────────────────────────────────────────────────────
IMG_H       = 224
IMG_W       = 1400
PATCH_SIZE  = 14
PATCH_H     = IMG_H // PATCH_SIZE   # 16
PATCH_W     = IMG_W // PATCH_SIZE   # 100
MASK_H      = PATCH_H * 4          # 64
MASK_W      = PATCH_W * 4          # 400
EMBED_DIM   = 768
NUM_CLASSES = 5
INTERMEDIATE_LAYERS = [3, 5, 8, 11]

# ── Данные (Эксп.13) ──────────────────────────────────────────────────
N_PER_CLASS  = 200
TEST_SIZE    = 0.2
SYNTH_WEIGHT = 0.5

# ── Fine-tuning (Эксп.16/17) ──────────────────────────────────────────
N_UNFREEZE   = 4
LR_BACKBONE  = 1e-5
LR_DECAY     = 0.75
LR_HEAD      = 1e-3
BATCH_SIZE   = 4
EPOCHS       = 75
FLIP_P       = 0.5
CROP_SCALE   = (0.85, 1.0)
BRIGHTNESS   = 0.3
USE_AMP      = (DEVICE == 'cuda')

# ── SSL v3 ────────────────────────────────────────────────────────────
SSL_BATCH_SIZE  = 4
SSL_EPOCHS      = 60    # v2 было 30
SSL_LR_BACKBONE = 1e-5
SSL_LR_PROJ     = 1e-3
SSL_TEMPERATURE = 0.5
SSL_PROJ_DIM    = 128
SSL_SAVE_LABEL  = 'exp18v3'

# Пути для resume
SSL_CKPT_PATH   = os.path.join(save_dir, f'ssl_checkpoint_{SSL_SAVE_LABEL}.pt')
V2_BACKBONE     = os.path.join(save_dir, 'dinov2_ssl_exp18v2.pt')

print(f'\nSSL batch: {SSL_BATCH_SIZE}  |  epochs: {SSL_EPOCHS}  |  label: {SSL_SAVE_LABEL}')
print(f'Warm-start backbone: {V2_BACKBONE}')
print(f'Resume checkpoint:   {SSL_CKPT_PATH}')

## 2. Загрузка данных

In [ ]:
train_df    = pd.read_csv(TRAIN_CSV)
labeled_ids = train_df['ImageId'].unique().tolist()
print(f'Строк: {len(train_df):,}  |  Размеченных изображений: {len(labeled_ids):,}')

all_train_imgs = sorted([f for f in os.listdir(TRAIN_IMG_DIR) if f.endswith('.jpg')])
print(f'Всего изображений в train_images: {len(all_train_imgs):,}  (используем для SSL)')


def decode_rle(rle_string, shape=(256, 1600)):
    if pd.isna(rle_string) or not isinstance(rle_string, str):
        return np.zeros(shape, dtype=np.uint8)
    nums   = list(map(int, rle_string.strip().split()))
    starts = np.array(nums[0::2]) - 1
    lens   = np.array(nums[1::2])
    mask   = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for s, l in zip(starts, lens):
        mask[s:s+l] = 1
    return mask.reshape(shape, order='F')


def build_segmask(image_id, df, shape=(256, 1600)):
    mask = np.zeros(shape, dtype=np.uint8)
    for _, row in df[df['ImageId'] == image_id].iterrows():
        cls = int(row['ClassId'])
        m   = decode_rle(row['EncodedPixels'], shape)
        mask[m == 1] = cls
    return mask


def compute_class_weights(image_ids, df, num_classes=NUM_CLASSES):
    px = Counter({c: 0 for c in range(num_classes)})
    for img_id in image_ids:
        mask = build_segmask(img_id, df)
        for c in range(num_classes):
            px[c] += int((mask == c).sum())
    total = sum(px.values())
    w = torch.tensor([total/(num_classes*(px[c]+1e-6)) for c in range(num_classes)])
    w = (w / w.mean()).clamp(min=0.1, max=5.0)
    for c, v in enumerate(w):
        print(f'  {"Фон" if c==0 else f"Дефект {c}"}: {v:.3f}  ({px[c]:,} пикс.)')
    return w.to(DEVICE)


classes_cache = train_df.groupby('ImageId')['ClassId'].apply(
    lambda x: sorted(x.dropna().astype(int).unique().tolist())
).to_dict()
label_map = {img_id: (cls_list[0] if cls_list else 0)
             for img_id, cls_list in classes_cache.items()}

print('Вспомогательные функции определены.')

## 3. Dataset (fine-tuning)

In [ ]:
class JointTransform:
    def __init__(self, img_h=IMG_H, img_w=IMG_W, is_train=True,
                 flip_p=FLIP_P, crop_scale=CROP_SCALE, brightness=BRIGHTNESS):
        self.img_h        = img_h
        self.img_w        = img_w
        self.is_train     = is_train
        self.flip_p       = flip_p
        self.crop_scale   = crop_scale
        self.color_jitter = transforms.ColorJitter(brightness=brightness)
        self.to_tensor    = transforms.ToTensor()
        self.normalize    = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __call__(self, img_pil, mask_np):
        img_pil  = img_pil.resize((self.img_w, self.img_h), Image.BILINEAR)
        mask_pil = Image.fromarray(mask_np).resize(
            (self.img_w, self.img_h), Image.NEAREST)
        if self.is_train:
            if random.random() < self.flip_p:
                img_pil  = img_pil.transpose(Image.FLIP_LEFT_RIGHT)
                mask_pil = mask_pil.transpose(Image.FLIP_LEFT_RIGHT)
            scale  = random.uniform(*self.crop_scale)
            crop_h = max(1, int(self.img_h * scale))
            crop_w = max(1, int(self.img_w * scale))
            top    = random.randint(0, self.img_h - crop_h)
            left   = random.randint(0, self.img_w - crop_w)
            img_pil  = img_pil.crop((left, top, left+crop_w, top+crop_h))
            mask_pil = mask_pil.crop((left, top, left+crop_w, top+crop_h))
            img_pil  = img_pil.resize((self.img_w, self.img_h), Image.BILINEAR)
            mask_pil = mask_pil.resize((self.img_w, self.img_h), Image.NEAREST)
            img_pil  = self.color_jitter(img_pil)
        img_t    = self.normalize(self.to_tensor(img_pil))
        mask_np2 = np.array(mask_pil, dtype=np.uint8)
        return img_t, mask_np2


class SteelSegDataset(Dataset):
    def __init__(self, ids, img_dir, df, joint_transform):
        self.ids=ids; self.img_dir=img_dir; self.df=df; self.jt=joint_transform
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img    = Image.open(os.path.join(self.img_dir, img_id)).convert('RGB')
        mask   = build_segmask(img_id, self.df)
        img_t, mask_np = self.jt(img, mask)
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).float()
        mask_t = F.interpolate(
            mask_t.unsqueeze(0), size=(MASK_H, MASK_W),
            mode='nearest').squeeze().long()
        return img_t, mask_t, 0


class SteelSegDatasetWithSynth(Dataset):
    def __init__(self, real_ids, img_dir, df, joint_transform, synth_pairs):
        self.real_ids = real_ids; self.img_dir = img_dir; self.df = df
        self.jt = joint_transform; self.synth = synth_pairs
        self.n_real = len(real_ids); self.n_synth = len(synth_pairs)
    def __len__(self): return self.n_real + self.n_synth
    def __getitem__(self, idx):
        if idx < self.n_real:
            img_id   = self.real_ids[idx]
            img      = Image.open(os.path.join(self.img_dir, img_id)).convert('RGB')
            mask     = build_segmask(img_id, self.df)
            is_synth = 0
        else:
            img_np, mask = self.synth[idx - self.n_real]
            img          = Image.fromarray(img_np.astype(np.uint8))
            is_synth     = 1
        img_t, mask_np = self.jt(img, mask)
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).float()
        mask_t = F.interpolate(
            mask_t.unsqueeze(0), size=(MASK_H, MASK_W),
            mode='nearest').squeeze().long()
        return img_t, mask_t, is_synth


train_jt = JointTransform(is_train=True)
val_jt   = JointTransform(is_train=False)
print('Datasets определены.')

## 4. Загрузка DINOv2

In [ ]:
DINOV2_LOCAL = 'C:/Users/MSI Katana 17/.cache/torch/hub/facebookresearch_dinov2_main'

dinov2 = torch.hub.load(
    DINOV2_LOCAL, 'dinov2_vitb14',
    pretrained=True, verbose=False, source='local'
)
dinov2 = dinov2.to(DEVICE)
for p in dinov2.parameters():
    p.requires_grad = False

n_blocks      = len(dinov2.blocks)        # 12
unfreeze_from = n_blocks - N_UNFREEZE     # 8
for i, block in enumerate(dinov2.blocks):
    if i >= unfreeze_from:
        for p in block.parameters():
            p.requires_grad = True
for p in dinov2.norm.parameters():
    p.requires_grad = True

frozen    = sum(p.numel() for p in dinov2.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in dinov2.parameters() if p.requires_grad)
print(f'DINOv2 ViT-B/14: {(frozen+trainable)/1e6:.1f}М параметров')
print(f'  Заморожено:  {frozen/1e6:.1f}М  (блоки 0–{unfreeze_from-1})')
print(f'  Разморожено: {trainable/1e6:.1f}М  (блоки {unfreeze_from}–{n_blocks-1} + norm)')

with torch.no_grad():
    test_img = torch.randn(1, 3, IMG_H, IMG_W).to(DEVICE)
    feats = dinov2.get_intermediate_layers(
        test_img, n=INTERMEDIATE_LAYERS, return_class_token=False)
    assert all(f.shape == (1, PATCH_H*PATCH_W, EMBED_DIM) for f in feats)
    print(f'Forward OK: {len(feats)} слоёв × {feats[0].shape}')
del test_img, feats

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM свободно: {free/1e9:.1f} / {total/1e9:.1f} Гб')

## 5. Эмбеддинги (из кэша) + KMeans-отбор

In [ ]:
emb_cache = os.path.join(save_dir, 'embeddings.pt')
print('Загружаем из кэша...')
ckpt_emb   = torch.load(emb_cache, map_location='cpu')
embeddings = ckpt_emb['embeddings']
emb_ids    = ckpt_emb['emb_ids']
emb_id_to_idx = {v: k for k, v in enumerate(emb_ids)}
print(f'Загружено: {embeddings.shape}')

all_labels = [label_map.get(i, 0) for i in labeled_ids]
train_ids, test_ids = train_test_split(
    labeled_ids, test_size=TEST_SIZE, stratify=all_labels, random_state=SEED)
print(f'Train pool: {len(train_ids):,}  |  Test: {len(test_ids):,}')


def kmeans_select(embeddings, image_ids, n_select, seed=SEED):
    km = KMeans(n_clusters=n_select, random_state=seed, n_init=10)
    km.fit(embeddings.numpy())
    selected = []
    for k in range(n_select):
        m = km.labels_ == k
        if not m.any(): continue
        center  = torch.tensor(km.cluster_centers_[k])
        ix      = np.where(m)[0]
        nearest = int(ix[torch.norm(embeddings[m] - center, dim=1).argmin().item()])
        selected.append(image_ids[nearest])
    return selected


def select_per_class(embeddings, image_ids, classes_cache,
                     n_per_class=N_PER_CLASS, seed=SEED):
    image_id_to_idx = {v: k for k, v in enumerate(image_ids)}
    selected_set    = set()
    for cls in [1, 2, 3, 4]:
        cls_ids = [i for i in image_ids if cls in classes_cache.get(i, [])]
        n_avail = len(cls_ids)
        if n_avail == 0: continue
        if n_avail <= n_per_class:
            chosen = cls_ids
            print(f'  Класс {cls}: {n_avail} доступно → берём все')
        else:
            idxs       = [image_id_to_idx[i] for i in cls_ids]
            cls_embeds = embeddings[idxs]
            chosen     = kmeans_select(cls_embeds, cls_ids, n_per_class, seed)
            print(f'  Класс {cls}: {n_avail} → {len(chosen)} (KMeans)')
        selected_set.update(chosen)
    result = list(selected_set)
    print(f'Итого уникальных: {len(result)}')
    return result


pool_mask   = [emb_id_to_idx[i] for i in train_ids]
pool_embeds = embeddings[pool_mask]
print('Отбор 200 изображений на класс:')
selected_ids = select_per_class(pool_embeds, train_ids, classes_cache)
assert len(set(selected_ids) & set(test_ids)) == 0, 'Пересечение с тест-сетом!'
print('Пересечений с тест-сетом: 0 ✓')

## 6. Архитектура: SegHeadDPT + Lovász-Softmax

In [ ]:
class SegHeadDPT(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, num_classes=NUM_CLASSES,
                 patch_h=PATCH_H, patch_w=PATCH_W, n_layers=4):
        super().__init__()
        self.patch_h  = patch_h
        self.patch_w  = patch_w
        self.n_layers = n_layers
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(embed_dim, 256, 1),
                          nn.BatchNorm2d(256), nn.GELU())
            for _ in range(n_layers)
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(256 * n_layers, 512, 1), nn.BatchNorm2d(512), nn.GELU(),
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.GELU())
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU())
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU())
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, features):
        maps = []
        for i, f in enumerate(features):
            B, N, C = f.shape
            x = f.reshape(B, self.patch_h, self.patch_w, C).permute(0,3,1,2)
            maps.append(self.proj[i](x))
        x = self.fuse(torch.cat(maps, dim=1))
        return self.head(self.up2(self.up1(x)))


def _lovász_grad(gt_sorted):
    n   = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union        = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard      = 1.0 - intersection / union
    if n > 1:
        jaccard[1:] = jaccard[1:] - jaccard[:-1]
    return jaccard


def _lovász_softmax_flat(probas, labels):
    C = probas.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if fg.sum() == 0: continue
        errors = (fg - probas[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        losses.append(torch.dot(errors_sorted, _lovász_grad(fg[perm])))
    return torch.stack(losses).mean() if losses else probas.sum() * 0.0


class LovászSoftmax(nn.Module):
    def forward(self, logits, targets):
        B, C, H, W = logits.shape
        probas  = F.softmax(logits.float(), dim=1).permute(0,2,3,1).reshape(-1, C)
        targets = targets.reshape(-1)
        return _lovász_softmax_flat(probas, targets)


_h = SegHeadDPT().to(DEVICE)
print(f'SegHeadDPT: {sum(p.numel() for p in _h.parameters()):,} параметров')
del _h

## 7. Синтетика (кэш Эксп.13)

In [ ]:
synth_cache = os.path.join(save_dir, 'synth_pairs_exp13.pkl')
print('Загружаем синтетику...')
with open(synth_cache, 'rb') as f:
    synth_pairs = pickle.load(f)
print(f'Загружено: {len(synth_pairs)} пар  |  Итого: {len(selected_ids)+len(synth_pairs)}')

## 8. Layer-wise LR decay

In [ ]:
def make_backbone_param_groups(model, base_lr=LR_BACKBONE, decay=LR_DECAY,
                                n_unfreeze=N_UNFREEZE, weight_decay=1e-2):
    nb = len(model.blocks)
    uf = nb - n_unfreeze
    groups = []
    norm_params = [p for p in model.norm.parameters() if p.requires_grad]
    if norm_params:
        groups.append({'params': norm_params, 'lr': base_lr,
                       'weight_decay': weight_decay, 'name': 'norm'})
    for i in range(nb - 1, uf - 1, -1):
        block_params = [p for p in model.blocks[i].parameters() if p.requires_grad]
        if not block_params: continue
        depth = nb - 1 - i
        lr_i  = base_lr * (decay ** depth)
        groups.append({'params': block_params, 'lr': lr_i,
                       'weight_decay': weight_decay, 'name': f'block_{i}'})
    return groups


print('Layer-wise LR (fine-tuning):')
for g in make_backbone_param_groups(dinov2):
    n_p = sum(p.numel() for p in g['params'])
    print(f'  {g["name"]:<10}  LR={g["lr"]:.2e}  params={n_p/1e6:.2f}М')

## 9. SSL Phase — SimCLR (v3: 60 эпох, warm-start, checkpoint resume)

### Логика запуска

```
Есть ssl_checkpoint_exp18v3.pt?  →  ДА  →  resume с последней эпохи
         ↓ НЕТ
Есть dinov2_ssl_exp18v2.pt?      →  ДА  →  warm-start с v2 (30 эп. SSL уже сделано)
         ↓ НЕТ
Старт с DINOv2 pretrained
```

Checkpoint сохраняется **каждую эпоху** — содержит backbone, optimizer, scheduler, history.

In [ ]:
class SSLAugmentation:
    """Аугментации для SimCLR: сохраняем горизонтальную структуру дефектов."""
    def __init__(self, img_h=IMG_H, img_w=IMG_W):
        self.img_h = img_h
        self.img_w = img_w
        self.jitter = transforms.ColorJitter(
            brightness=0.4, contrast=0.4, saturation=0.2, hue=0.05)
        self.blur = transforms.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __call__(self, img_pil):
        img = img_pil.resize((self.img_w, self.img_h), Image.BILINEAR)
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        scale  = random.uniform(0.75, 1.0)
        crop_h = max(1, int(self.img_h * scale))
        crop_w = max(1, int(self.img_w * scale))
        top    = random.randint(0, self.img_h - crop_h)
        left   = random.randint(0, self.img_w - crop_w)
        img    = img.crop((left, top, left+crop_w, top+crop_h))
        img    = img.resize((self.img_w, self.img_h), Image.BILINEAR)
        img    = self.jitter(img)
        if random.random() < 0.5:
            img = self.blur(img)
        return self.normalize(self.to_tensor(img))


class SimCLRDataset(Dataset):
    def __init__(self, image_ids, img_dir, augmentation):
        self.ids  = image_ids
        self.dir  = img_dir
        self.aug  = augmentation
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.dir, self.ids[idx])).convert('RGB')
        return self.aug(img), self.aug(img)


class ProjectionHead(nn.Module):
    """SimCLR projection head: 2-layer MLP с BN и L2-нормализацией выхода."""
    def __init__(self, in_dim=EMBED_DIM, hidden_dim=512, out_dim=SSL_PROJ_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


def nt_xent_loss(z1, z2, temperature=SSL_TEMPERATURE):
    """NT-Xent (SimCLR) loss. z1, z2: (B, D) нормализованные эмбеддинги."""
    B  = z1.size(0)
    z  = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.T) / temperature
    sim.fill_diagonal_(float('-inf'))
    labels = torch.cat([torch.arange(B, 2*B), torch.arange(B)]).to(z.device)
    return F.cross_entropy(sim, labels)


print('SSL классы определены.')
_ph = ProjectionHead().to(DEVICE)
print(f'ProjectionHead: {sum(p.numel() for p in _ph.parameters()):,} параметров')
del _ph

## 10. Функция SSL-обучения с resume

In [ ]:
def train_ssl_v3(all_image_ids, img_dir,
                 n_epochs=SSL_EPOCHS, batch_size=SSL_BATCH_SIZE,
                 label=SSL_SAVE_LABEL):
    """
    SSL-фаза SimCLR с полным checkpoint-resume.
    Приоритет загрузки:
      1. ssl_checkpoint_{label}.pt  — продолжаем с последней эпохи
      2. dinov2_ssl_exp18v2.pt      — warm-start с v2 backbone
      3. DINOv2 pretrained           — старт с нуля
    """
    ckpt_path   = os.path.join(save_dir, f'ssl_checkpoint_{label}.pt')
    best_path   = os.path.join(save_dir, f'dinov2_ssl_{label}.pt')
    v2_bb_path  = os.path.join(save_dir, 'dinov2_ssl_exp18v2.pt')

    # Отдельная копия backbone для SSL (не портим основной dinov2)
    bb = copy.deepcopy(dinov2).to(DEVICE)
    for p in bb.parameters(): p.requires_grad = False
    for i in range(n_blocks - N_UNFREEZE, n_blocks):
        for p in bb.blocks[i].parameters(): p.requires_grad = True
    for p in bb.norm.parameters(): p.requires_grad = True

    proj = ProjectionHead().to(DEVICE)

    ssl_ds = SimCLRDataset(all_image_ids, img_dir, SSLAugmentation())
    ssl_dl = DataLoader(ssl_ds, batch_size=batch_size, shuffle=True,
                        num_workers=0, drop_last=True)

    bb_grps  = make_backbone_param_groups(bb, base_lr=SSL_LR_BACKBONE,
                                          decay=LR_DECAY, weight_decay=1e-4)
    proj_grp = [{'params': list(proj.parameters()),
                 'lr': SSL_LR_PROJ, 'weight_decay': 1e-4, 'name': 'proj'}]
    opt   = optim.AdamW(bb_grps + proj_grp)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    hist        = {'ssl_loss': []}
    best_loss   = float('inf')
    start_epoch = 1

    # ── Приоритет 1: resume из v3 checkpoint ──────────────────────────
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        bb.load_state_dict(ckpt['backbone'])
        proj.load_state_dict(ckpt['projection_head'])
        opt.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        if scaler and 'scaler' in ckpt:
            scaler.load_state_dict(ckpt['scaler'])
        hist        = ckpt['history']
        best_loss   = ckpt['best_loss']
        start_epoch = ckpt['epoch'] + 1
        print(f'▶ Resume с эпохи {start_epoch}/{n_epochs}  '
              f'(лучший loss: {best_loss:.5f})')

    # ── Приоритет 2: warm-start с v2 backbone ─────────────────────────
    elif os.path.exists(v2_bb_path):
        bb.load_state_dict(torch.load(v2_bb_path, map_location=DEVICE))
        print(f'▶ Warm-start: загружен backbone v2  ({v2_bb_path})')
        print(f'  SSL v2 уже сделал 30 эпох → продолжаем с 60')

    # ── Приоритет 3: старт с нуля ─────────────────────────────────────
    else:
        print('▶ Старт с DINOv2 pretrained (v2 backbone не найден)')

    print('='*65)
    print(f'SSL v3 | {n_epochs} эп. | batch={batch_size} | '
          f'T={SSL_TEMPERATURE} | AMP={USE_AMP}')
    print(f'  Изображений: {len(ssl_ds):,}  |  Батчей/эпоху: {len(ssl_dl)}')
    print('='*65)

    for ep in range(start_epoch, n_epochs + 1):
        bb.eval()
        for i in range(n_blocks - N_UNFREEZE, n_blocks):
            bb.blocks[i].train()
        bb.norm.train()
        proj.train()

        total_loss = 0.0
        for v1, v2 in ssl_dl:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            opt.zero_grad()

            if scaler:
                with torch.amp.autocast('cuda'):
                    cls1 = bb.forward_features(v1)['x_norm_clstoken']
                    cls2 = bb.forward_features(v2)['x_norm_clstoken']
                    z1   = proj(cls1)
                    z2   = proj(cls2)
                    loss = nt_xent_loss(z1, z2)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                cls1 = bb.forward_features(v1)['x_norm_clstoken']
                cls2 = bb.forward_features(v2)['x_norm_clstoken']
                z1   = proj(cls1)
                z2   = proj(cls2)
                loss = nt_xent_loss(z1, z2)
                loss.backward()
                opt.step()

            total_loss += loss.item()

        sched.step()
        avg = total_loss / len(ssl_dl)
        hist['ssl_loss'].append(avg)

        # Сохраняем лучший backbone
        if avg < best_loss:
            best_loss = avg
            torch.save(bb.state_dict(), best_path)

        # Полный checkpoint каждую эпоху (для resume)
        ckpt_data = {
            'epoch':           ep,
            'backbone':        bb.state_dict(),
            'projection_head': proj.state_dict(),
            'optimizer':       opt.state_dict(),
            'scheduler':       sched.state_dict(),
            'history':         hist,
            'best_loss':       best_loss,
        }
        if scaler:
            ckpt_data['scaler'] = scaler.state_dict()
        torch.save(ckpt_data, ckpt_path)

        if ep % 5 == 0 or ep == 1:
            used = torch.cuda.memory_allocated()/1e9 if DEVICE == 'cuda' else 0
            print(f'[SSL v3] Эп {ep:3d}/{n_epochs} | '
                  f'Loss: {avg:.5f} | Best: {best_loss:.5f} | VRAM: {used:.1f} Гб')

    # Загружаем лучший backbone
    bb.load_state_dict(torch.load(best_path, map_location=DEVICE))
    with open(os.path.join(save_dir, f'history_ssl_{label}.json'), 'w') as f:
        json.dump(hist, f)
    print(f'\nSSL v3 завершён. Лучший loss: {best_loss:.5f}  '
          f'(эп. {hist["ssl_loss"].index(min(hist["ssl_loss"]))+1})')
    return bb, hist


print('SSL функция определена.')

## 11. Запуск SSL (60 эпох)

In [ ]:
import time as _time

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM свободно: {free/1e9:.1f} / {total/1e9:.1f} Гб')

_ssl_t0 = _time.perf_counter()
dinov2_ssl_v3, ssl_hist_v3 = train_ssl_v3(all_train_imgs, TRAIN_IMG_DIR)
ssl_train_time_sec = _time.perf_counter() - _ssl_t0

print(f'Время SSL-фазы: {ssl_train_time_sec/60:.1f} мин. '
      f'({ssl_train_time_sec/SSL_EPOCHS:.1f} сек./эпоха)')

## 12. SSL convergence: v1 / v2 / v3

In [ ]:
ssl_histories = {}
for name, fname in [
    ('v1 (batch=2, 15 эп.)',  'history_ssl_exp18.json'),
    ('v2 (batch=4, 30 эп.)',  'history_ssl_exp18v2.json'),
]:
    p = os.path.join(save_dir, fname)
    if os.path.exists(p):
        with open(p) as f:
            ssl_histories[name] = json.load(f)['ssl_loss']

ssl_histories['v3 (batch=4, 60 эп.)'] = ssl_hist_v3['ssl_loss']

fig, ax = plt.subplots(figsize=(12, 5))
colors_ssl = {'v1 (batch=2, 15 эп.)': '#9E9E9E',
              'v2 (batch=4, 30 эп.)': '#FF9800',
              'v3 (batch=4, 60 эп.)': '#E53935'}
for name, losses in ssl_histories.items():
    lw = 2.5 if 'v3' in name else 1.5
    ax.plot(range(1, len(losses)+1), losses,
            color=colors_ssl[name], lw=lw,
            label=f'{name}  best={min(losses):.5f}')

ax.set_xlabel('Эпоха'); ax.set_ylabel('NT-Xent Loss')
ax.set_title('SSL convergence: v1 / v2 / v3', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

for name, losses in ssl_histories.items():
    print(f'{name}: start={losses[0]:.5f}  best={min(losses):.5f}  '
          f'last={losses[-1]:.5f}')

## 13. Fine-tuning (пайплайн Эксп.17)

In [ ]:
def train_finetune_with_ssl(ssl_backbone, train_ids, val_ids, df,
                             synth_pairs=None, n_epochs=EPOCHS,
                             label='ft_exp18v3', checkpoint_every=25):
    """Fine-tuning идентичен Эксп.17, backbone загружен из SSL."""
    for p in ssl_backbone.parameters(): p.requires_grad = False
    for i in range(n_blocks - N_UNFREEZE, n_blocks):
        for p in ssl_backbone.blocks[i].parameters(): p.requires_grad = True
    for p in ssl_backbone.norm.parameters(): p.requires_grad = True

    head = SegHeadDPT().to(DEVICE)

    bb_groups  = make_backbone_param_groups(ssl_backbone)
    head_group = [{'params': list(head.parameters()),
                   'lr': LR_HEAD, 'weight_decay': 1e-4, 'name': 'head'}]
    opt   = optim.AdamW(bb_groups + head_group)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    print('Веса классов:')
    cw   = compute_class_weights(train_ids, df)
    cce  = nn.CrossEntropyLoss(weight=cw, reduction='none')
    clov = LovászSoftmax()

    if synth_pairs is not None:
        tds = SteelSegDatasetWithSynth(
            train_ids, TRAIN_IMG_DIR, df, train_jt, synth_pairs)
    else:
        tds = SteelSegDataset(train_ids, TRAIN_IMG_DIR, df, train_jt)
    vds = SteelSegDataset(val_ids, TRAIN_IMG_DIR, df, val_jt)
    tdl = DataLoader(tds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    vdl = DataLoader(vds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    iou  = JaccardIndex(task='multiclass', num_classes=NUM_CLASSES,
                        average='none').to(DEVICE)
    hist = {'train_loss': [], 'val_miou': [], 'val_iou_per_class': []}
    best = 0.0
    best_state_head = None
    best_state_bb   = None

    print(f'Обучающих: {len(tds)}  |  Валидационных: {len(vds)}')
    print(f'AMP: {USE_AMP}  |  Batch: {BATCH_SIZE}  |  Батчей/эпоху: {len(tdl)}')

    for ep in range(1, n_epochs + 1):
        ssl_backbone.eval()
        for i in range(n_blocks - N_UNFREEZE, n_blocks):
            ssl_backbone.blocks[i].train()
        ssl_backbone.norm.train()
        head.train()

        tl = 0.0
        for imgs, masks, is_synth in tdl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    feats = ssl_backbone.get_intermediate_layers(
                        imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                    lg = head(feats)
                    px_loss = cce(lg, masks)
                    w_batch = torch.where(
                        is_synth.bool().to(DEVICE)[:, None, None],
                        torch.full_like(px_loss, SYNTH_WEIGHT),
                        torch.ones_like(px_loss))
                    loss = (px_loss * w_batch).mean() + clov(lg, masks)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                feats = ssl_backbone.get_intermediate_layers(
                    imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                lg = head(feats)
                px_loss = cce(lg, masks)
                w_batch = torch.where(
                    is_synth.bool().to(DEVICE)[:, None, None],
                    torch.full_like(px_loss, SYNTH_WEIGHT),
                    torch.ones_like(px_loss))
                loss = (px_loss * w_batch).mean() + clov(lg, masks)
                loss.backward()
                opt.step()
            tl += loss.item()
        sched.step()

        ssl_backbone.eval(); head.eval(); iou.reset()
        with torch.no_grad():
            for imgs, masks, _ in vdl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                if USE_AMP:
                    with torch.amp.autocast('cuda'):
                        feats = ssl_backbone.get_intermediate_layers(
                            imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                        preds = head(feats).argmax(1)
                else:
                    feats = ssl_backbone.get_intermediate_layers(
                        imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                    preds = head(feats).argmax(1)
                iou.update(preds, masks)

        ipc = iou.compute().cpu().numpy()
        mi  = float(ipc.mean())
        hist['train_loss'].append(tl / len(tdl))
        hist['val_miou'].append(mi)
        hist['val_iou_per_class'].append(ipc.tolist())

        if mi > best:
            best = mi
            best_state_head = {k: v.clone() for k, v in head.state_dict().items()}
            best_state_bb   = {k: v.clone() for k, v in ssl_backbone.state_dict().items()}

        if ep % 10 == 0 or ep == 1:
            s = '  '.join([f'cls{i}:{v:.3f}' for i, v in enumerate(ipc)])
            print(f'[{label}] Эп {ep:3d}/{n_epochs} | '
                  f'Loss:{tl/len(tdl):.4f} | mIoU:{mi:.4f} | {s}')

        if ep % checkpoint_every == 0:
            ckpt_path = os.path.join(save_dir, f'ckpt_{label}_ep{ep}.pt')
            torch.save({'epoch': ep, 'head_state': head.state_dict(),
                        'backbone_state': ssl_backbone.state_dict(),
                        'history': hist, 'best_miou': best}, ckpt_path)
            print(f'  Чекпоинт: {ckpt_path}')

    head.load_state_dict(best_state_head)
    ssl_backbone.load_state_dict(best_state_bb)
    print(f'\n  -> Лучший mIoU: {best:.4f}')
    return head, hist


print('Fine-tuning функция определена.')

## 14. Запуск fine-tuning

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM свободно: {free/1e9:.1f} / {total/1e9:.1f} Гб')

print('='*70)
print(f'Эксп.18 v3 — Fine-tuning ({EPOCHS} эпох, пайплайн Эксп.17)')
print(f'  Backbone: SSL v3 (60 эп. SimCLR, warm-start с v2)')
print(f'  Данные: {len(selected_ids)} реал. + {len(synth_pairs)} синт.')
print(f'  Loss: CE(weighted) + Lovász | AMP: {USE_AMP}')
print('='*70)

_ft_t0 = _time.perf_counter()
model_18v3, history_18v3 = train_finetune_with_ssl(
    dinov2_ssl_v3, selected_ids, test_ids, train_df,
    synth_pairs=synth_pairs, label='ft_exp18v3'
)
ft_train_time_sec = _time.perf_counter() - _ft_t0

print(f'\nВремя fine-tuning: {ft_train_time_sec/60:.1f} мин.')
print(f'Суммарное время (SSL + FT): {(ssl_train_time_sec+ft_train_time_sec)/60:.1f} мин.')

## 15. Итоговое сравнение: Эксп.17 / 18 v1 / 18 v2 / 18 v3

In [ ]:
histories = {}
for name, fname in [
    ('Эксп.17 (без SSL)',          'history_exp17.json'),
    ('Эксп.18 v1 (SSL batch=2)',   'history_exp18.json'),
    ('Эксп.18 v2 (SSL batch=4, 30 эп.)', 'history_exp18v2.json'),
]:
    p = os.path.join(save_dir, fname)
    if os.path.exists(p):
        with open(p) as f: histories[name] = json.load(f)

histories['Эксп.18 v3 (SSL batch=4, 60 эп.)'] = history_18v3

names_ordered = [
    'Эксп.17 (без SSL)',
    'Эксп.18 v1 (SSL batch=2)',
    'Эксп.18 v2 (SSL batch=4, 30 эп.)',
    'Эксп.18 v3 (SSL batch=4, 60 эп.)',
]
colors = ['#2196F3', '#9E9E9E', '#FF9800', '#E53935']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name, color in zip(names_ordered, colors):
    if name not in histories: continue
    h  = histories[name]
    mi = max(h['val_miou'])
    ep = int(np.argmax(h['val_miou'])) + 1
    lw = 2.5 if 'v3' in name else 1.5
    axes[0].plot(h['val_miou'], color=color, lw=lw,
                 label=f'{name}\nbest={mi:.4f} (эп.{ep})')

axes[0].set_title('Val mIoU по эпохам', fontsize=13)
axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('mIoU')
axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.4, 0.75)

bar_names = [n for n in names_ordered if n in histories]
bar_vals  = [max(histories[n]['val_miou']) for n in bar_names]
bar_cols  = [c for n, c in zip(names_ordered, colors) if n in histories]

bars = axes[1].bar(range(len(bar_vals)), bar_vals,
                   color=bar_cols, width=0.5, zorder=2)
axes[1].set_xticks(range(len(bar_names)))
axes[1].set_xticklabels([n.split('(')[0].strip() for n in bar_names],
                         fontsize=9, rotation=15)
axes[1].set_ylabel('Лучший mIoU')
axes[1].set_title('Итоговое сравнение', fontsize=13)
axes[1].set_ylim(min(bar_vals) - 0.02, max(bar_vals) + 0.02)
axes[1].grid(True, axis='y', alpha=0.3, zorder=1)
for bar, val in zip(bars, bar_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001, f'{val:.4f}',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Эксп.18 v3: SimCLR 60 эп. (warm-start с v2)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

# Сводная таблица
print('\nСводная таблица:')
print(f'{"Эксперимент":<40} {"mIoU":>8} {"Δ к Эксп.17":>12}')
print('-' * 62)
base = max(histories.get('Эксп.17 (без SSL)', {}).get('val_miou', [0]))
for name in names_ordered:
    if name not in histories: continue
    mi    = max(histories[name]['val_miou'])
    delta = f'{(mi-base)*100:+.2f}%' if base > 0 else '—'
    print(f'{name:<40} {mi:>8.4f} {delta:>12}')

cls_names = ['Фон', 'Дефект 1', 'Дефект 2', 'Дефект 3', 'Дефект 4']
print('\nIoU по классам (лучшая эпоха):')
avail = [n for n in names_ordered if n in histories]
print(f'{"Класс":<12}', end='')
for n in avail: print(f'{n[:14]:>16}', end='')
print()
print('-' * (12 + 16 * len(avail)))
for c, cname in enumerate(cls_names):
    print(f'{cname:<12}', end='')
    for n in avail:
        h  = histories[n]
        ep = int(np.argmax(h['val_miou']))
        print(f'{h["val_iou_per_class"][ep][c]:>16.4f}', end='')
    print()

## 16. Сохранение

In [ ]:
torch.save(model_18v3.state_dict(),
           os.path.join(save_dir, 'model_exp18v3.pt'))
torch.save(dinov2_ssl_v3.state_dict(),
           os.path.join(save_dir, 'dinov2_exp18v3.pt'))
with open(os.path.join(save_dir, 'history_exp18v3.json'), 'w') as f:
    json.dump(history_18v3, f)

m_v3 = max(history_18v3['val_miou'])
e_v3 = int(np.argmax(history_18v3['val_miou'])) + 1
i_v3 = history_18v3['val_iou_per_class'][e_v3 - 1]

print('Сохранено:')
print('  model_exp18v3.pt         — голова DPT')
print('  dinov2_exp18v3.pt        — backbone (после SSL v3 + fine-tuning)')
print('  dinov2_ssl_exp18v3.pt    — backbone только после SSL v3')
print('  history_exp18v3.json     — история fine-tuning')
print('  history_ssl_exp18v3.json — история SSL')

print(f'\nЛучший mIoU Эксп.18 v3: {m_v3:.4f}  (эпоха {e_v3})')
print(f'IoU по классам: {[f"{v:.3f}" for v in i_v3]}')

for cmp_name, cmp_key in [
    ('Эксп.17 (без SSL)',          'history_exp17.json'),
    ('Эксп.18 v2 (SSL 30 эп.)',    'history_exp18v2.json'),
]:
    p = os.path.join(save_dir, cmp_key)
    if os.path.exists(p):
        with open(p) as f: cmp_h = json.load(f)
        cmp_m = max(cmp_h['val_miou'])
        print(f'Прирост vs {cmp_name}: {(m_v3-cmp_m)*100:+.2f}%')